# 11｜标准感知到避障

本课使用合成 LiDAR 点云，不连接设备，也不发送运动命令。

![协同与智能能力](assets/advanced.svg)


## FleetDock 避障管线

标准传感器样本交给 Provider，Provider 生成标准障碍物，`AvoidanceManager` 再按策略评估路径。
点云降采样、聚类、TTL 和障碍物快照都由 SDK 处理。


In [ ]:
from array import array
from types import SimpleNamespace
import time

from fleetdock import (
    CoordinateFrame,
    LidarObstacleConfig,
    LidarObstacleProvider,
    PointCloudFrame,
    SensorDescriptor,
    SensorKind,
    SensorMount,
    Vector3,
)

descriptor = SensorDescriptor(
    sensor_id="teaching-lidar",
    kind=SensorKind.LIDAR,
    frame=CoordinateFrame.BODY,
    mount=SensorMount(position_m=Vector3(0.0, 0.0, 0.0)),
)
pose = SimpleNamespace(
    local_x_m=0.0,
    local_y_m=0.0,
    local_z_m=-10.0,
    local_position_frame=CoordinateFrame.LOCAL_NED,
    local_position_updated_at_s=time.monotonic(),
    roll_deg=0.0,
    pitch_deg=0.0,
    yaw_deg=0.0,
)
provider = LidarObstacleProvider(
    descriptor,
    pose_provider=lambda: pose,
    config=LidarObstacleConfig(),
)


In [ ]:
points = array("f", [
    5.0, 0.0, 0.0,
    5.2, 0.2, 0.1,
    8.0, 3.0, 0.0,
])
provider.observe(
    PointCloudFrame(
        sensor_id="teaching-lidar",
        kind=SensorKind.LIDAR,
        sequence=1,
        frame=CoordinateFrame.BODY,
        data=memoryview(points),
        point_count=3,
    )
)

print("Provider 就绪：", provider.wait_until_ready(2.0))
print("Provider 状态：", provider.status())
print("标准障碍物：", provider.snapshot())
provider.close()


## 把 Provider 接入设备

真实设备中只需：

```python
drone.avoidance.add_provider(provider)
drone.avoidance.configure(
    AvoidancePolicy(strategy=AvoidanceStrategy.LOCAL_REPLAN)
)
plan.route(route, avoidance=drone.avoidance.policy).run()
```

传感器解析、障碍物聚合、风险判断、局部绕行和降级动作由 SDK 串联。
